In [0]:
# Mount ADLS Gen2
# Required each time the cluster is restarted which should be only on the first notebook as they run in order

tiers = ["bronze", "silver", "gold"]
adls_paths = {tier: f"abfss://{tier}@adlsnikhildev.dfs.core.windows.net/" for tier in tiers}

# Accessing paths
bronze_adls = adls_paths["bronze"]
silver_adls = adls_paths["silver"]
gold_adls = adls_paths["gold"] 

print(dbutils.fs.ls(bronze_adls))
print(dbutils.fs.ls(silver_adls))
print(dbutils.fs.ls(gold_adls))

[FileInfo(path='abfss://bronze@adlsnikhildev.dfs.core.windows.net/2026-06-05_earthquake_data.json', name='2026-06-05_earthquake_data.json', size=427152, modificationTime=1780731273000)]
[FileInfo(path='abfss://silver@adlsnikhildev.dfs.core.windows.net/earthquake_events_silver/', name='earthquake_events_silver/', size=0, modificationTime=1780732173000)]
[]


In [0]:
# from datetime import date, timedelta
# start_date = date.today() - timedelta(days=1)
# end_date = date.today()

In [0]:
try:
    # Running inside a Databricks Workflow

    bronze_output = dbutils.jobs.taskValues.get(
        taskKey="bronze",
        key="bronze_output"
    )

    silver_output_path = dbutils.jobs.taskValues.get(
        taskKey="silver",
        key="silver_output"
    )

except Exception as e:
    print(f"Running outside Workflow: {e}")

    # Development / classroom fallback

    bronze_output = {
        "start_date": "2026-06-01",
        "end_date": "2026-06-06",
        "bronze_path": adls_paths["bronze"],
        "silver_path": adls_paths["silver"],
        "gold_path": adls_paths["gold"]
    }

    silver_output_path = (
        f"{adls_paths['silver']}earthquake_events_silver/"
    )

# Extract values
start_date = bronze_output["start_date"]
end_date = bronze_output["end_date"]
gold_adls = bronze_output["gold_path"]

print(f"Silver Path: {silver_output_path}")
print(f"Gold Path: {gold_adls}")

Running outside Workflow: Must pass debugValue when calling get outside of a job context. debugValue cannot be None.
Silver Path: abfss://silver@adlsnikhildev.dfs.core.windows.net/earthquake_events_silver/
Gold Path: abfss://gold@adlsnikhildev.dfs.core.windows.net/


In [0]:
!pip install reverse_geocoder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for reverse_geocoder: filename=reverse_geocoder-1.5.1-py3-none-any.whl size=2268069 sha256=e5b9db8ab2f249e937b193400dff6282842422719a3c2ed94ed2b54da345ad1b
  Stored in directory: /home/spark-12493620-bba4-4983-903e-20/.cache/pip/wheels/11/e1/67/6e47f0ad41ea1843d37e1fbe79c6074744a1f4aace641cf800
Successfully built reverse_geocoder
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
# Ensure the below library is installed on your cluster
import reverse_geocoder as rg
from datetime import date, timedelta

df = spark.read.parquet(silver_output_path).filter(col('time') > start_date)

df = df.limit(100)  # added to speed up processing as during testing it was proving a bottleneck
# The problem is caused by the Python UDF (reverse_geocoder) being a bottleneck due to its non-parallel nature and high computational cost


In [0]:
def get_country_code(lat, lon):
    """
    Retrieve the country code for a given latitude and longitude.
    
    Parameters:
        lat (float or str): Latitude of the location.
        lon (float or str): Longitude of the location.
    
    Returns:
        str: Country code of the location, retrieved using the reverse geocoding API.
    """
    try:
        coordinates = (float(lat), float(lon))
        result = rg.search(coordinates)[0].get('cc')
        print(f"Processed coordinates: {coordinates} -> {result}")
        return result
    except Exception as e:
        print(f"Error processing coordinates: {lat}, {lon} -> {str(e)}")
        return None

# Example:
# >> get_country_details(48.8588443, 2.2943506)
# 'FR'

In [0]:
get_country_code(48.8588443, 2.2943506)

Loading formatted geocoded file...
Processed coordinates: (48.8588443, 2.2943506) -> FR


'FR'

In [0]:
get_country_code_udf = udf(get_country_code, StringType())
df_with_location = df.withColumn('country_code', get_country_code_udf(col('latitude'), col('longitude')))

In [0]:

# adding significance classification
df_with_location_sig_class = \
                            df_with_location.\
                                withColumn('sig_class', 
                                            when(col("sig") < 100, "Low").\
                                            when((col("sig") >= 100) & (col("sig") < 500), "Moderate").\
                                            otherwise("High")
                                            )

In [0]:

df_with_location_sig_class.printSchema()

root
 |-- id: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- elevation: double (nullable = true)
 |-- title: string (nullable = true)
 |-- place_description: string (nullable = true)
 |-- sig: long (nullable = true)
 |-- mag: double (nullable = true)
 |-- magType: string (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- updated: timestamp (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sig_class: string (nullable = false)



In [0]:

# Save the transformed DataFrame to the Silver container
gold_output_path = f"{gold_adls}earthquake_events_gold/"

In [0]:

# Append DataFrame to Silver container in Parquet format
df_with_location_sig_class.write.mode('append').parquet(gold_output_path)
     